<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Task в C#, который будет представлять задачи внутри
проекта. На основе этого класса разработать 2-3 производных класса,
демонстрирующих принципы наследования и полиморфизма. В каждом из классов
должны быть реализованы новые атрибуты и методы, а также переопределены
некоторые методы базового класса для демонстрации полиморфизма

Требования к базовому классу Task:

• Атрибуты: ID задачи (TaskId), Название задачи (TaskName), Приоритет
задачи (Priority).

Методы:

o MarkAsComplete(): метод для отметки задачи как выполненной.

o GetTaskDetails(): метод для получения деталей задачи.

o ReassignTo(): метод для переназначения задачи другому члену
команды.

Требования к производным классам:
1. ДелегатскаяЗадача (DelegateTask): Должна содержать дополнительные
атрибуты, такие как Дата выполнения (DueDate).
Метод MarkAsComplete() должен быть переопределен для включения даты
выполнения в сообщение о завершении задачи.
2. КоманднаяЗадача (TeamTask): Должна содержать дополнительные атрибуты,
такие как Команда (TeamName). Метод ReassignTo() должен быть
переопределен для указания нового члена команды, которому будет
переназначена задача.
3. ИсследовательскаяЗадача (ResearchTask) (если требуется третий класс):
Должна содержать дополнительные атрибуты, такие как Исходные данные
(DataSource). Метод GetTaskDetails() должен быть переопределен для
отображения источников данных, используемых в задаче, вместе с другими
деталями задачи.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [6]:
using System;
using System.Collections.Generic;
using System.Linq;

// ==== Интерфейсы ====
public interface ITrackable
{
    void ShowStatus();
}

public interface IPrioritizable
{
    void UpgradePriority(string newPriority);
}

public interface ITaskService
{
    void AddTask(Task task);
    void DisplayAllTasks();
    Task? FindById(int id);
}

// ==== Делегаты и события ====
public delegate void TaskStatusChangedHandler(Task task, string message);
public delegate void TaskAddedHandler(Task task);

// ==== Базовый класс Task ====
public class Task : ITrackable, IPrioritizable
{
    public int TaskId { get; set; }
    public string TaskName { get; set; }
    public string Priority { get; set; }
    public string AssignedTo { get; private set; }
    public bool IsCompleted { get; private set; }

    public DateTime CreatedDate { get; set; }
    public string Description { get; set; }
    public int EstimatedHours { get; set; }
    public string Category { get; set; }
    public decimal Budget { get; set; }
    public List<string> Tags { get; set; }

    public event TaskStatusChangedHandler? StatusChanged;

    public Task(int taskId, string taskName, string priority, string assignedTo = "Не назначено")
    {
        TaskId = taskId;
        TaskName = taskName;
        Priority = priority;
        AssignedTo = assignedTo;
        IsCompleted = false;
        CreatedDate = DateTime.Now;
        Description = "";
        EstimatedHours = 0;
        Category = "Общее";
        Budget = 0;
        Tags = new List<string>();
    }

    public void AddTag(string tag)
    {
        Tags.Add(tag);
        StatusChanged?.Invoke(this, $"Тег '{tag}' добавлен к задаче '{TaskName}'");
    }

    public virtual void MarkAsComplete()
    {
        IsCompleted = true;
        StatusChanged?.Invoke(this, $"Задача '{TaskName}' выполнена");
    }

    public virtual void ReassignTo(string newMember)
    {
        AssignedTo = newMember;
        StatusChanged?.Invoke(this, $"Задача '{TaskName}' переназначена на {AssignedTo}");
    }

    public virtual void UpgradePriority(string newPriority)
    {
        Priority = newPriority;
        StatusChanged?.Invoke(this, $"Приоритет задачи '{TaskName}' обновлён на {Priority}");
    }

    public void ShowStatus()
    {
        Console.WriteLine($"Статус задачи '{TaskName}': {(IsCompleted ? "Выполнена" : "В процессе")}");
    }

    public virtual void GetTaskDetails()
    {
        Console.WriteLine($"ID:{TaskId}, Название:{TaskName}, Приоритет:{Priority}, Назначена:{AssignedTo}, Статус:{(IsCompleted ? "Да" : "Нет")}, Теги:[{string.Join(", ", Tags)}]");
    }

    // Защищенный метод для вызова события из производных классов
    protected void OnStatusChanged(string message)
    {
        StatusChanged?.Invoke(this, message);
    }
}

// ==== Производные классы ====
public class DelegateTask : Task
{
    public string DelegateTo { get; set; }
    public DateTime DueDate { get; set; }

    public DelegateTask(int taskId, string taskName, string priority, DateTime dueDate, string assignedTo = "Не назначено")
        : base(taskId, taskName, priority, assignedTo)
    {
        DelegateTo = "Не назначено";
        DueDate = dueDate;
    }

    public void Delegate(string member)
    {
        DelegateTo = member;
        OnStatusChanged($"Задача '{TaskName}' делегирована на {DelegateTo}");
    }

    public override void GetTaskDetails()
    {
        base.GetTaskDetails();
        Console.WriteLine($"   Делегирована: {DelegateTo}, Срок: {DueDate.ToShortDateString()}");
    }
}

public class TeamTask : Task
{
    public string TeamName { get; set; }
    public List<string> TeamMembers { get; set; }

    public TeamTask(int taskId, string taskName, string priority, string teamName, string assignedTo = "Не назначено")
        : base(taskId, taskName, priority, assignedTo)
    {
        TeamName = teamName;
        TeamMembers = new List<string>();
    }

    public void AddTeamMember(string member)
    {
        TeamMembers.Add(member);
        OnStatusChanged($"В команду '{TeamName}' добавлен {member}");
    }

    public override void GetTaskDetails()
    {
        base.GetTaskDetails();
        Console.WriteLine($"   Команда: {TeamName}, Участники: [{string.Join(", ", TeamMembers)}]");
    }
}

public class ResearchTask : Task
{
    public string DataSource { get; set; }
    public string Method { get; set; }
    public List<string> References { get; set; }

    public ResearchTask(int taskId, string taskName, string priority, string dataSource, string method, string assignedTo = "Не назначено")
        : base(taskId, taskName, priority, assignedTo)
    {
        DataSource = dataSource;
        Method = method;
        References = new List<string>();
    }

    public void AddReference(string reference)
    {
        References.Add(reference);
        OnStatusChanged($"Добавлена ссылка '{reference}' к исследованию '{TaskName}'");
    }

    public override void GetTaskDetails()
    {
        base.GetTaskDetails();
        Console.WriteLine($"   Источник: {DataSource}, Метод: {Method}, Ссылки: [{string.Join(", ", References)}]");
    }
}

// ==== Generic-класс коллекции ====
public class TaskCollection<T> where T : Task
{
    private readonly List<T> _tasks = new List<T>();

    public event TaskAddedHandler? TaskAdded;

    public void Add(T task)
    {
        _tasks.Add(task);
        TaskAdded?.Invoke(task);
    }

    public T? Find(Predicate<T> predicate) => _tasks.Find(predicate);

    public void DisplayTasks() => _tasks.ForEach(t => t.GetTaskDetails());

    public List<T> GetHighPriorityTasks() => _tasks.Where(t => t.Priority.Equals("Высокий", StringComparison.OrdinalIgnoreCase)).ToList();
}

// ==== Сервис для DI ====
public class TaskService : ITaskService
{
    private readonly TaskCollection<Task> _collection;

    public TaskService(TaskCollection<Task> collection)
    {
        _collection = collection;
    }

    public void AddTask(Task task) => _collection.Add(task);

    public void DisplayAllTasks() => _collection.DisplayTasks();

    public Task? FindById(int id) => _collection.Find(t => t.TaskId == id);
}

// ==== Создание и использование задач с лямбда-подпиской на события ====
var task1 = new Task(1, "Обычная задача", "Низкий", "Иван");
var task2 = new DelegateTask(2, "Сделать отчёт", "Высокий", DateTime.Now.AddDays(3), "Анна");
var task3 = new TeamTask(3, "Разработка модуля", "Средний", "Команда А", "Мария");
var task4 = new ResearchTask(4, "Анализ рынка", "Высокий", "Open Data", "Статистика", "Алексей");

TaskStatusChangedHandler lambdaHandler = (task, message) => Console.WriteLine($"[Лямбда] {message}");

// Подписка на события через лямбду
task1.StatusChanged += lambdaHandler;
task2.StatusChanged += lambdaHandler;
task3.StatusChanged += lambdaHandler;
task4.StatusChanged += lambdaHandler;

// Создание коллекции и подписка на событие TaskAdded через лямбду
var allTasks = new TaskCollection<Task>();
allTasks.TaskAdded += task => Console.WriteLine($"[Лямбда] Задача '{task.TaskName}' добавлена в коллекцию");

// Добавление задач
allTasks.Add(task1);
allTasks.Add(task2);
allTasks.Add(task3);
allTasks.Add(task4);

// Демонстрация работы событий
task1.MarkAsComplete();
task2.ReassignTo("Сергей");
task3.UpgradePriority("Высокий");
((DelegateTask)task2).Delegate("Анна");
((TeamTask)task3).AddTeamMember("Ольга");
((ResearchTask)task4).AddReference("Market Report 2025");

// LINQ: выбор задач с высоким приоритетом
Console.WriteLine("\n=== Задачи с высоким приоритетом ===");
allTasks.GetHighPriorityTasks().ForEach(t => t.GetTaskDetails());

[Лямбда] Задача 'Обычная задача' добавлена в коллекцию
[Лямбда] Задача 'Сделать отчёт' добавлена в коллекцию
[Лямбда] Задача 'Разработка модуля' добавлена в коллекцию
[Лямбда] Задача 'Анализ рынка' добавлена в коллекцию
[Лямбда] Задача 'Обычная задача' выполнена
[Лямбда] Задача 'Сделать отчёт' переназначена на Сергей
[Лямбда] Приоритет задачи 'Разработка модуля' обновлён на Высокий
[Лямбда] Задача 'Сделать отчёт' делегирована на Анна
[Лямбда] В команду 'Команда А' добавлен Ольга
[Лямбда] Добавлена ссылка 'Market Report 2025' к исследованию 'Анализ рынка'

=== Задачи с высоким приоритетом ===
ID:2, Название:Сделать отчёт, Приоритет:Высокий, Назначена:Сергей, Статус:Нет, Теги:[]
   Делегирована: Анна, Срок: 11/20/2025
ID:3, Название:Разработка модуля, Приоритет:Высокий, Назначена:Мария, Статус:Нет, Теги:[]
   Команда: Команда А, Участники: [Ольга]
ID:4, Название:Анализ рынка, Приоритет:Высокий, Назначена:Алексей, Статус:Нет, Теги:[]
   Источник: Open Data, Метод: Статистика, Ссылки: [Mar